# Hub Perturbation Inputs, Flip Rate / McNemar Test, and Heatmap

**Sections:**
1. Build ROI names + AAL-to-Yeo7 network mapping (shared across everything below)
2. Hub identification (degree-based top-20 hubs per subject)
3. Clinical vs. random hub perturbation -- generates `hub_clinical_perturbed_k{K}.json` / `hub_random_perturbed_k{K}.json`
4. Network-labeled variant -- adds a `"network"` field, generates the `*_wnetwork*.json` files
5. Flip Rate + McNemar's Test across all 6 models / 3 formats / 2 K values
6. Combined heatmap figure (3-panel: Baseline vs Clinical, Random vs Clinical, Baseline vs Random)


## 1. ROI names + AAL-to-Yeo7 network mapping

In [1]:
import os
import glob
import json
import numpy as np
import pandas as pd
import bct
from xml.etree import ElementTree
from sklearn.utils import Bunch
from nilearn import datasets, image as _image

CODE_DIR = "/home/siu856622573/Code"
RQ4_DIR = "/home/siu856622573/Code/ADHD code/RQ4"
INPUTS_DIR = os.path.join(RQ4_DIR, "inputs")
os.makedirs(INPUTS_DIR, exist_ok=True)

AAL_LOCAL_DIR = os.path.join(CODE_DIR, "aal")
N_ROIS = 116

ROI_NAMES_JSON = os.path.join(CODE_DIR, "aal_roi_names.json")           # shared legend, read by the RQ4 experiment notebooks
EDGE_LIST_NAMES = os.path.join(CODE_DIR, "edge_list_top10pos_names.json")   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
HUB_GROUND_TRUTH_20ROI = os.path.join(CODE_DIR, "hub_ground_truth_20roi.json")


def load_local_aal(base_dir=AAL_LOCAL_DIR):
    """Load the AAL atlas from a local directory (instead of downloading from nilearn)."""
    nii_candidates = sorted(glob.glob(os.path.join(base_dir, "**", "*.nii"), recursive=True))
    xml_candidates = sorted(glob.glob(os.path.join(base_dir, "**", "*.xml"), recursive=True))

    nii_path = next((p for p in nii_candidates if "aal" in os.path.basename(p).lower()),
                     nii_candidates[0] if nii_candidates else None)
    xml_path = next((p for p in xml_candidates if "aal" in os.path.basename(p).lower()),
                     xml_candidates[0] if xml_candidates else None)

    if nii_path is None or xml_path is None:
        raise FileNotFoundError(
            f"Could not find AAL .nii/.xml under {base_dir}. "
            f"Found .nii: {nii_candidates}, .xml: {xml_candidates}"
        )

    labels = ["Background"]
    indices = ["0"]
    xml_tree = ElementTree.parse(xml_path)
    root = xml_tree.getroot()
    for label in root.iter("label"):
        indices.append(label.find("index").text)
        labels.append(label.find("name").text)

    return Bunch(maps=nii_path, labels=labels, indices=indices)


def build_aal_roi_names():
    """Returns a list of 116 AAL region names, ordered to match ROI_NETWORK / the FC matrix
    columns (ascending AAL label value -- same convention used in build_aal_to_yeo7 and the
    gradient-visualization cells)."""
    aal = load_local_aal()
    aal_img = _image.load_img(aal.maps)
    aal_data = aal_img.get_fdata()
    aal_values = np.unique(aal_data)
    aal_values = aal_values[aal_values != 0]

    label_lookup = {int(idx): lab for idx, lab in zip(aal.indices, aal.labels)}
    names = [label_lookup[int(v)] for v in aal_values]
    return names


ROI_NAMES = build_aal_roi_names()
assert len(ROI_NAMES) == N_ROIS

with open(ROI_NAMES_JSON, "w") as f:
    json.dump(ROI_NAMES, f, indent=2)

print(f"Saved {len(ROI_NAMES)} ROI names -> {ROI_NAMES_JSON}")


Saved 116 ROI names -> /home/siu856622573/Code/aal_roi_names.json


In [2]:
yeo = datasets.fetch_atlas_yeo_2011()
print(yeo.keys())


[fetch_atlas_yeo_2011] Dataset found in /home/siu856622573/nilearn_data/yeo_2011

dict_keys(['maps', 'labels', 'description', 'lut', 'atlas_type', 'template', 'anat'])


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

YEO7_NAMES = {
    1: "Vis", 2: "SomMot", 3: "DorsAttn", 4: "SalVentAttn",
    5: "Limbic", 6: "Cont", 7: "Default",
}


def build_aal_to_yeo7():
    """Assign each of the 116 AAL ROIs (in .1D column order) to one of Yeo's
    7 networks, via majority-vote voxel overlap with the resampled Yeo-7 atlas."""
    aal = load_local_aal()
    aal_img = _image.load_img(aal.maps)
    aal_data = aal_img.get_fdata()
    aal_values = np.unique(aal_data)
    aal_values = aal_values[aal_values != 0]

    yeo_img = _image.load_img(yeo.maps)
    yeo_resampled = _image.resample_to_img(yeo_img, aal_img, interpolation="nearest")
    yeo_data = yeo_resampled.get_fdata().squeeze()

    roi_network = []
    for val in aal_values:
        mask = aal_data == val
        vox = yeo_data[mask]
        vox = vox[vox > 0]
        if vox.size == 0:
            roi_network.append(None)
            continue
        labels, counts = np.unique(vox, return_counts=True)
        roi_network.append(int(labels[np.argmax(counts)]))
    return roi_network


ROI_NETWORK = build_aal_to_yeo7()
assert len(ROI_NETWORK) == N_ROIS

NETWORKS = sorted(set(n for n in ROI_NETWORK if n is not None))

print("ROI counts per Yeo-7 network:")
print(pd.Series([YEO7_NAMES.get(n, "Unassigned") for n in ROI_NETWORK]).value_counts())

# String-label version for the hub/perturbation pipeline below: it expects
# roi_network as a list of NAME strings ("Vis", "Default", ...), not the
# integer Yeo labels (1-7) ROI_NETWORK stores.
roi_network = [YEO7_NAMES.get(n, "Unassigned") for n in ROI_NETWORK]

n_unassigned = roi_network.count("Unassigned")
if n_unassigned > 0:
    print(f"\nWARNING: {n_unassigned} ROI(s) had no Yeo-7 overlap and are labeled 'Unassigned'.")
    print("These are automatically excluded from CLINICAL_NETWORKS matching below --")
    print("not treated as a real 8th network. If this count is large, double-check")
    print("AAL/Yeo atlas alignment before trusting the perturbation-condition split.")

NETWORK_NAMES = sorted(set(roi_network) - {"Unassigned"})
print(f"\nNETWORK_NAMES: {NETWORK_NAMES}")


ROI counts per Yeo-7 network:
Unassigned     22
Vis            22
Limbic         20
Default        19
SomMot         13
Cont           10
SalVentAttn     8
DorsAttn        2
Name: count, dtype: int64

These are automatically excluded from CLINICAL_NETWORKS matching below --
not treated as a real 8th network. If this count is large, double-check
AAL/Yeo atlas alignment before trusting the perturbation-condition split.

NETWORK_NAMES: ['Cont', 'Default', 'DorsAttn', 'Limbic', 'SalVentAttn', 'SomMot', 'Vis']


## 2. Hub identification -- config + degree computation

In [16]:
# =============================================================================
# Config -- set K_SWAP to 5, run sections 2-4, then set it to 10 and re-run
# sections 2-4 to produce both K=5 and K=10 input files.
# =============================================================================
TOP_N_HUBS = 20          # size of the hub list (unchanged across both conditions)
K_SWAP = 10                # number of hubs swapped out per subject (K <= TOP_N_HUBS)
SEED = 0

# Networks treated as "clinically important" for ADHD (Yeo-7 labels built above)
CLINICAL_NETWORKS = ("Default", "Cont", "SalVentAttn")

with open(ROI_NAMES_JSON) as f:
    roi_names = json.load(f)
assert len(roi_names) == N_ROIS

with open(EDGE_LIST_NAMES) as f:
    edge_list_names_all = json.load(f)

_name_to_idx = {name: i for i, name in enumerate(roi_names)}
print(f"Loaded {len(roi_names)} ROI names, {len(edge_list_names_all)} subjects.")


Loaded 116 ROI names, 768 subjects.


## Parsing + degree computation (unchanged from the existing pipeline)

In [17]:
def parse_edge_section_only(text):
    """text = the full saved string ('Node feature:\n...\n\nEdge feature:\n...').
    Returns just the edge lines; the Node feature section is never read."""
    edge_block = text.split("Edge feature:\n", 1)[1]
    return [line for line in edge_block.strip().splitlines() if line]


def parse_named_edge_line(line):
    """'RegionA:weight:RegionB' -> (name_i, weight, name_j). Positive-only enforced."""
    name_i, w, name_j = line.split(":")
    w = float(w)
    assert w > 0, f"Non-positive edge weight found ({w}) -- input must be positive-only (FC > 0)."
    return (name_i, w, name_j)


def edges_for_subject(subject_id):
    text = edge_list_names_all[subject_id]
    edge_lines = parse_edge_section_only(text)
    return [parse_named_edge_line(line) for line in edge_lines]


def binary_adjacency_from_named_edges(edges, name_to_idx, n_rois):
    A = np.zeros((n_rois, n_rois))
    for (name_i, w, name_j) in edges:
        i, j = name_to_idx[name_i], name_to_idx[name_j]
        A[i, j] = 1
        A[j, i] = 1
    return A


def compute_degree_hubs_from_json(edge_list_names_all, roi_names, top_n=TOP_N_HUBS):
    subject_ids = list(edge_list_names_all.keys())
    n_subjects = len(subject_ids)
    degrees_all = np.zeros((n_subjects, N_ROIS))

    # Step 1: degree for every ROI, every subject
    for s, sid in enumerate(subject_ids):
        edges = edges_for_subject(sid)
        A = binary_adjacency_from_named_edges(edges, _name_to_idx, N_ROIS)
        degrees_all[s] = bct.degrees_und(A)

    degree_z = (degrees_all - degrees_all.mean(axis=0)) / (degrees_all.std(axis=0) + 1e-8)

    hub_results = {}
    rank_order_full = {}   # full 116-ROI rank order per subject, big -> small degree (needed for non-hub pool)
    for s, sid in enumerate(subject_ids):
        # Step 2: sort all 116 ROIs by degree, descending
        # Step 3: keep top_n (20) -- ties broken by |z| so it's still deterministic
        order = np.lexsort((-np.abs(degree_z[s]), -degrees_all[s]))
        hub_idx = order[:top_n]
        rank_order_full[sid] = order   # full ranking, index 0 = highest degree

        hub_results[sid] = sorted(
            [
                {"roi": roi_names[i], "degree": int(degrees_all[s, i]), "cohort_z": float(degree_z[s, i])}
                for i in hub_idx
            ],
            key=lambda x: x["cohort_z"], reverse=True
        )
    return hub_results, subject_ids, rank_order_full, degrees_all, degree_z


hub_results, subject_ids, rank_order_full, degrees_all, degree_z = compute_degree_hubs_from_json(edge_list_names_all, roi_names, top_n=TOP_N_HUBS)
print(f"Computed hubs for {len(hub_results)} subjects, {TOP_N_HUBS} hubs each.")

example_sid = subject_ids[0]
print(f"\nExample ({example_sid}), top 5 hubs:")
for h in hub_results[example_sid][:5]:
    print(" ", h)
print(f"Hub list length: {len(hub_results[example_sid])} (should be {TOP_N_HUBS})")


Computed hubs for 768 subjects, 20 hubs each.

Example (1018959), top 5 hubs:
  {'roi': 'Cingulum_Post_R', 'degree': 22, 'cohort_z': 2.7523148352243627}
  {'roi': 'Parietal_Sup_L', 'degree': 20, 'cohort_z': 2.683022361167184}
  {'roi': 'Temporal_Pole_Mid_L', 'degree': 19, 'cohort_z': 1.7847711032116718}
  {'roi': 'Parietal_Sup_R', 'degree': 17, 'cohort_z': 1.7815097482585058}
  {'roi': 'Frontal_Sup_Orb_L', 'degree': 20, 'cohort_z': 1.7543198922436116}
Hub list length: 20 (should be 20)


## Non-hub pool

"Non-hub" = ROI ranked **below** the top-20 for that subject (rank 21-116), i.e. NOT already
in the hub list. Both perturbation conditions draw replacements only from this pool.

In [18]:
def get_non_hub_pool(sid, roi_names, rank_order_full, top_n=TOP_N_HUBS):
    """Return the ROIs ranked below the hub cutoff (lowest degree) as non-hub candidates."""
    order = rank_order_full[sid]
    non_hub_idx = order[top_n:]   # everything below rank 20
    return [roi_names[i] for i in non_hub_idx]


# sanity check
non_hub_example = get_non_hub_pool(example_sid, roi_names, rank_order_full)
print(f"Non-hub pool size for {example_sid}: {len(non_hub_example)} (should be {N_ROIS - TOP_N_HUBS})")
print("First 5 non-hub ROIs:", non_hub_example[:5])


Non-hub pool size for 1018959: 96 (should be 96)
First 5 non-hub ROIs: ['Paracentral_Lobule_R', 'Temporal_Inf_R', 'Frontal_Sup_L', 'Frontal_Inf_Oper_L', 'Frontal_Inf_Tri_L']


## 3. Clinical vs. random hub perturbation

**Clinical:** rank the current top-20 hubs by whether they belong to a clinical network
(`CLINICAL_NETWORKS`), then swap out the K *most* clinically-relevant hubs for K random
non-hub ROIs. If fewer than K clinical hubs exist, fall back to the remaining highest-rank
hubs so K is always met (keeps K fixed across subjects, matching the random condition
exactly on swap count).

**Random:** swap out K randomly chosen hubs (from the full 20, no network filter) for K
random non-hub ROIs. Same K, same replacement pool, only *which* hubs are chosen differs.

In [19]:
def perturb_hubs_clinical(hub_list, non_hub_pool, roi_network, name_to_idx, degrees_all_row, degree_z_row,
                           k=K_SWAP, seed=SEED):
    rng = np.random.default_rng(seed)
    hub_list = list(hub_list)  # don't mutate original

    def is_clinical(entry):
        return roi_network[name_to_idx[entry["roi"]]] in CLINICAL_NETWORKS

    # highest cohort_z clinical hubs first; if <k clinical hubs exist, fill remainder
    # with the next-highest-z non-clinical hubs so k is always satisfied
    clinical_hubs_sorted = sorted([h for h in hub_list if is_clinical(h)], key=lambda h: h["cohort_z"], reverse=True)
    other_hubs_sorted = sorted([h for h in hub_list if not is_clinical(h)], key=lambda h: h["cohort_z"], reverse=True)

    swap_out = (clinical_hubs_sorted + other_hubs_sorted)[:k]
    swap_out_names = {h["roi"] for h in swap_out}

    kept = [h for h in hub_list if h["roi"] not in swap_out_names]
    swap_in_names = rng.choice(non_hub_pool, size=k, replace=False)
    swapped_in = [
        {
            "roi": name,
            "degree": int(degrees_all_row[name_to_idx[name]]),
            "cohort_z": float(degree_z_row[name_to_idx[name]]),
        }
        for name in swap_in_names
    ]
    perturbed = kept + swapped_in
    assert len(perturbed) == len(hub_list), "Hub list size must stay fixed."
    return perturbed, [h["roi"] for h in swap_out], list(swap_in_names)


def perturb_hubs_random(hub_list, non_hub_pool, degrees_all_row, degree_z_row, name_to_idx,
                         k=K_SWAP, seed=SEED):
    rng = np.random.default_rng(seed)
    hub_list = list(hub_list)

    swap_out_idx = rng.choice(len(hub_list), size=k, replace=False)
    swap_out_names = {hub_list[i]["roi"] for i in swap_out_idx}

    kept = [h for h in hub_list if h["roi"] not in swap_out_names]
    swap_in_names = rng.choice(non_hub_pool, size=k, replace=False)
    swapped_in = [
        {
            "roi": name,
            "degree": int(degrees_all_row[name_to_idx[name]]),
            "cohort_z": float(degree_z_row[name_to_idx[name]]),
        }
        for name in swap_in_names
    ]

    perturbed = kept + swapped_in
    assert len(perturbed) == len(hub_list), "Hub list size must stay fixed."
    return perturbed, [hub_list[i]["roi"] for i in swap_out_idx], list(swap_in_names)


## Run for all subjects, save both files (for the current `K_SWAP`)

In [20]:
hub_clinical_perturbed = {}
hub_random_perturbed = {}
swap_log = {}   # exact ROIs swapped, per subject/condition -- useful for later analysis/debugging
print("swap_out = hubs that are removed \nswap_in = non-hubs that are added instead")

for s, sid in enumerate(subject_ids):
    hub_list = hub_results[sid]
    non_hub_pool = get_non_hub_pool(sid, roi_names, rank_order_full)

    clinical_hubs, c_out, c_in = perturb_hubs_clinical(
        hub_list, non_hub_pool, roi_network, _name_to_idx,
        degrees_all_row=degrees_all[s], degree_z_row=degree_z[s], k=K_SWAP, seed=SEED
    )
    random_hubs, r_out, r_in = perturb_hubs_random(
        hub_list, non_hub_pool, degrees_all_row=degrees_all[s], degree_z_row=degree_z[s],
        name_to_idx=_name_to_idx, k=K_SWAP, seed=SEED
    )

    hub_clinical_perturbed[sid] = clinical_hubs
    hub_random_perturbed[sid] = random_hubs
    swap_log[sid] = {
        "clinical": {"swapped_out": c_out, "swapped_in": c_in},
        "random": {"swapped_out": r_out, "swapped_in": r_in},
    }

clinical_path = os.path.join(INPUTS_DIR, f"hub_clinical_perturbed_k{K_SWAP}.json")
random_path = os.path.join(INPUTS_DIR, f"hub_random_perturbed_k{K_SWAP}.json")
swap_log_path = os.path.join(CODE_DIR, f"hub_perturbation_swap_log_k{K_SWAP}.json")

with open(clinical_path, "w") as f:
    json.dump(hub_clinical_perturbed, f, indent=2)
with open(random_path, "w") as f:
    json.dump(hub_random_perturbed, f, indent=2)
with open(swap_log_path, "w") as f:
    json.dump(swap_log, f, indent=2)

print(f"Saved {len(hub_clinical_perturbed)} subjects -> {clinical_path}")
print(f"Saved {len(hub_random_perturbed)} subjects -> {random_path}")
print(f"Saved swap log -> {swap_log_path}")


swap_out = hubs that are removed 
swap_in = non-hubs that are added instead
Saved 768 subjects -> /home/siu856622573/Code/ADHD code/RQ4/inputs/hub_clinical_perturbed_k10.json
Saved 768 subjects -> /home/siu856622573/Code/ADHD code/RQ4/inputs/hub_random_perturbed_k10.json
Saved swap log -> /home/siu856622573/Code/hub_perturbation_swap_log_k10.json


## Sanity check

In [21]:
print(f"=== {example_sid} ===")
print(f"\nOriginal hub list size: {len(hub_results[example_sid])}")
print(f"Clinical-perturbed size: {len(hub_clinical_perturbed[example_sid])}  (must equal {TOP_N_HUBS})")
print(f"Random-perturbed size:   {len(hub_random_perturbed[example_sid])}  (must equal {TOP_N_HUBS})")

print(f"\nClinical condition -- swapped OUT ({K_SWAP}):", swap_log[example_sid]["clinical"]["swapped_out"])
print(f"Clinical condition -- swapped IN  ({K_SWAP}):", swap_log[example_sid]["clinical"]["swapped_in"])

print(f"\nRandom condition -- swapped OUT ({K_SWAP}):", swap_log[example_sid]["random"]["swapped_out"])
print(f"Random condition -- swapped IN  ({K_SWAP}):", swap_log[example_sid]["random"]["swapped_in"])

# Swap-in ROIs must come from the non-hub pool (overlap between clinical and
# random swap-out is fine -- both start from the same 20 hubs)
non_hub_pool_check = set(get_non_hub_pool(example_sid, roi_names, rank_order_full))
assert set(swap_log[example_sid]["clinical"]["swapped_in"]).issubset(non_hub_pool_check)
assert set(swap_log[example_sid]["random"]["swapped_in"]).issubset(non_hub_pool_check)
print("\nAll swapped-in ROIs confirmed to come from the non-hub pool (rank > 20).")


=== 1018959 ===

Original hub list size: 20
Clinical-perturbed size: 20  (must equal 20)
Random-perturbed size:   20  (must equal 20)

Clinical condition -- swapped OUT (10): ['Cingulum_Post_R', 'Supp_Motor_Area_L', 'Cingulum_Post_L', 'Frontal_Sup_Medial_L', 'SupraMarginal_L', 'Frontal_Sup_Medial_R', 'Insula_L', 'Frontal_Med_Orb_L', 'Frontal_Med_Orb_R', 'Parietal_Sup_L']
Clinical condition -- swapped IN  (10): [np.str_('Caudate_L'), np.str_('Cerebelum_7b_R'), np.str_('Rectus_L'), np.str_('Cerebelum_Crus2_R'), np.str_('Vermis_3'), np.str_('Frontal_Inf_Oper_L'), np.str_('Temporal_Inf_R'), np.str_('Angular_L'), np.str_('SupraMarginal_R'), np.str_('Frontal_Inf_Oper_R')]

Random condition -- swapped OUT (10): ['SupraMarginal_L', 'Temporal_Sup_R', 'Cingulum_Post_L', 'Supp_Motor_Area_L', 'Parietal_Sup_R', 'Cingulum_Post_R', 'Cerebelum_Crus1_L', 'Frontal_Sup_Orb_L', 'Temporal_Pole_Sup_R', 'Parietal_Sup_L']
Random condition -- swapped IN  (10): [np.str_('Frontal_Inf_Oper_L'), np.str_('Paracentr

## 4. Network-labeled hub files

Adds a `"network"` field (Yeo-7 label) to each hub entry, for three parallel files (per `K_SWAP`):
1. `hub_not_perturbed_wnetwork.json` -- baseline top-20 hubs + network label (K-independent)
2. `hub_clinical_perturbed_wnetwork_k{K_SWAP}.json` -- clinical swap + network label
3. `hub_random_perturbed_wnetwork_k{K_SWAP}.json` -- random swap + network label

A separate verification cell cross-checks ROI names between `hub_ground_truth_20roi.json` and
the new `hub_not_perturbed_wnetwork.json`, per subject, and flags any mismatch.

In [22]:
def add_network_field(hub_list, roi_network, name_to_idx):
    return [{**h, "network": roi_network[name_to_idx[h["roi"]]]} for h in hub_list]


def perturb_hubs_clinical_net(hub_list, non_hub_pool, roi_network, name_to_idx,
                               degrees_all_row, degree_z_row, k=K_SWAP, seed=SEED):
    rng = np.random.default_rng(seed)
    hub_list = list(hub_list)

    def is_clinical(entry):
        return roi_network[name_to_idx[entry["roi"]]] in CLINICAL_NETWORKS

    clinical_hubs_sorted = sorted([h for h in hub_list if is_clinical(h)],
                                   key=lambda h: h["cohort_z"], reverse=True)
    other_hubs_sorted = sorted([h for h in hub_list if not is_clinical(h)],
                                key=lambda h: h["cohort_z"], reverse=True)

    swap_out = (clinical_hubs_sorted + other_hubs_sorted)[:k]
    swap_out_names = {h["roi"] for h in swap_out}

    kept = [{**h, "network": roi_network[name_to_idx[h["roi"]]]}
            for h in hub_list if h["roi"] not in swap_out_names]

    swap_in_names = rng.choice(non_hub_pool, size=k, replace=False)
    swapped_in = [
        {"roi": name, "degree": int(degrees_all_row[name_to_idx[name]]),
         "cohort_z": float(degree_z_row[name_to_idx[name]]),
         "network": roi_network[name_to_idx[name]]}
        for name in swap_in_names
    ]

    perturbed = kept + swapped_in
    assert len(perturbed) == len(hub_list), "Hub list size must stay fixed."
    return perturbed, [h["roi"] for h in swap_out], list(swap_in_names)


def perturb_hubs_random_net(hub_list, non_hub_pool, degrees_all_row, degree_z_row, name_to_idx,
                             k=K_SWAP, seed=SEED):
    rng = np.random.default_rng(seed)
    hub_list = list(hub_list)

    swap_out_idx = rng.choice(len(hub_list), size=k, replace=False)
    swap_out_names = {hub_list[i]["roi"] for i in swap_out_idx}

    kept = [{**h, "network": roi_network[name_to_idx[h["roi"]]]}
            for h in hub_list if h["roi"] not in swap_out_names]

    swap_in_names = rng.choice(non_hub_pool, size=k, replace=False)
    swapped_in = [
        {"roi": name, "degree": int(degrees_all_row[name_to_idx[name]]),
         "cohort_z": float(degree_z_row[name_to_idx[name]]),
         "network": roi_network[name_to_idx[name]]}
        for name in swap_in_names
    ]

    perturbed = kept + swapped_in
    assert len(perturbed) == len(hub_list), "Hub list size must stay fixed."
    return perturbed, [hub_list[i]["roi"] for i in swap_out_idx], list(swap_in_names)


In [23]:
hub_clinical_perturbed_wnetwork = {}
hub_random_perturbed_wnetwork = {}
swap_log_wnetwork = {}

for s, sid in enumerate(subject_ids):
    hub_list = hub_results[sid]
    non_hub_pool = get_non_hub_pool(sid, roi_names, rank_order_full)

    clinical_hubs, c_out, c_in = perturb_hubs_clinical_net(
        hub_list, non_hub_pool, roi_network, _name_to_idx,
        degrees_all_row=degrees_all[s], degree_z_row=degree_z[s], k=K_SWAP, seed=SEED
    )
    random_hubs, r_out, r_in = perturb_hubs_random_net(
        hub_list, non_hub_pool, degrees_all_row=degrees_all[s], degree_z_row=degree_z[s],
        name_to_idx=_name_to_idx, k=K_SWAP, seed=SEED
    )
    hub_clinical_perturbed_wnetwork[sid] = clinical_hubs
    hub_random_perturbed_wnetwork[sid] = random_hubs

    swap_log_wnetwork[sid] = {
        "clinical": {"swapped_out": c_out, "swapped_in": c_in},
        "random": {"swapped_out": r_out, "swapped_in": r_in},
    }

clinical_net_path = os.path.join(INPUTS_DIR, f"hub_clinical_perturbed_wnetwork_k{K_SWAP}.json")
random_net_path = os.path.join(INPUTS_DIR, f"hub_random_perturbed_wnetwork_k{K_SWAP}.json")
not_perturbed_net_path = os.path.join(INPUTS_DIR, "hub_not_perturbed_wnetwork.json")

with open(clinical_net_path, "w") as f:
    json.dump(hub_clinical_perturbed_wnetwork, f, indent=2)
with open(random_net_path, "w") as f:
    json.dump(hub_random_perturbed_wnetwork, f, indent=2)

hub_not_perturbed_wnetwork = {
    sid: add_network_field(hub_results[sid], roi_network, _name_to_idx)
    for sid in subject_ids
}
with open(not_perturbed_net_path, "w") as f:
    json.dump(hub_not_perturbed_wnetwork, f, indent=2)

print(f"Saved {len(hub_clinical_perturbed_wnetwork)} subjects -> {clinical_net_path}")
print(f"Saved {len(hub_random_perturbed_wnetwork)} subjects -> {random_net_path}")
print(f"Saved {len(hub_not_perturbed_wnetwork)} subjects -> {not_perturbed_net_path}")

print(f"\nExample ({example_sid}), first 3 not-perturbed (network-labeled) hubs:")
for h in hub_not_perturbed_wnetwork[example_sid][:3]:
    print(" ", h)
print(f"\nExample ({example_sid}), first 3 clinical-perturbed (network-labeled) hubs:")
for h in hub_clinical_perturbed_wnetwork[example_sid][:3]:
    print(" ", h)


Saved 768 subjects -> /home/siu856622573/Code/ADHD code/RQ4/inputs/hub_clinical_perturbed_wnetwork_k10.json
Saved 768 subjects -> /home/siu856622573/Code/ADHD code/RQ4/inputs/hub_random_perturbed_wnetwork_k10.json
Saved 768 subjects -> /home/siu856622573/Code/ADHD code/RQ4/inputs/hub_not_perturbed_wnetwork.json

Example (1018959), first 3 not-perturbed (network-labeled) hubs:
  {'roi': 'Cingulum_Post_R', 'degree': 22, 'cohort_z': 2.7523148352243627, 'network': 'Default'}
  {'roi': 'Parietal_Sup_L', 'degree': 20, 'cohort_z': 2.683022361167184, 'network': 'DorsAttn'}
  {'roi': 'Temporal_Pole_Mid_L', 'degree': 19, 'cohort_z': 1.7847711032116718, 'network': 'Limbic'}

Example (1018959), first 3 clinical-perturbed (network-labeled) hubs:
  {'roi': 'Temporal_Pole_Mid_L', 'degree': 19, 'cohort_z': 1.7847711032116718, 'network': 'Limbic'}
  {'roi': 'Parietal_Sup_R', 'degree': 17, 'cohort_z': 1.7815097482585058, 'network': 'DorsAttn'}
  {'roi': 'Frontal_Sup_Orb_L', 'degree': 20, 'cohort_z': 1.7

## Verification: `hub_ground_truth_20roi.json` vs. `hub_not_perturbed_wnetwork.json`

Both should contain the SAME 20 ROI names per subject (network field is the only
addition) -- flags any subject where the ROI sets don't match exactly.

In [24]:
with open(HUB_GROUND_TRUTH_20ROI) as f:
    check_gt = json.load(f)
with open(os.path.join(INPUTS_DIR, "hub_not_perturbed_wnetwork.json")) as f:
    check_wn = json.load(f)

mismatches = []
for sid in check_gt:
    gt_names = {h["roi"] for h in check_gt[sid]}
    wn_names = {h["roi"] for h in check_wn[sid]}
    if gt_names != wn_names:
        mismatches.append(sid)

print(f"Checked {len(check_gt)} subjects for ground-truth vs. network-labeled ROI match.")
print(f"Mismatches: {len(mismatches)}" + (f" -> {mismatches[:5]}..." if mismatches else " (none)"))

size_errors = [
    sid for sid in subject_ids
    if len(hub_clinical_perturbed_wnetwork[sid]) != TOP_N_HUBS
    or len(hub_random_perturbed_wnetwork[sid]) != TOP_N_HUBS
]
print(f"Perturbed-file size errors: {len(size_errors)}" + (f" -> {size_errors[:5]}..." if size_errors else " (none)"))


Checked 768 subjects for ground-truth vs. network-labeled ROI match.
Mismatches: 0 (none)
Perturbed-file size errors: 0 (none)


## 5. Flip Rate and McNemar's Test

In [28]:
import os
import json
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from collections import Counter

GT_PATH = os.path.join(CODE_DIR, "subject_summary.csv")

def load_ground_truth(csv_path):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "ADHD"}  # DX_binary
    return {str(row["subject_id"]): label_map[int(row["DX_binary"])] for _, row in df.iterrows()}

gt = load_ground_truth(GT_PATH)
VALID_LABELS = {"ADHD", "Control"}
print(Counter(gt.values()))
BASE_DIR = "/home/siu856622573/Code/RQ_1"
# ============================================================
# Model directories
# ============================================================
MODEL_DIRS = {
    "DeepSeek-V4": os.path.join(BASE_DIR, "deepseek_v4"),
    "R1-Distill": os.path.join(BASE_DIR, "deepseek-r1-distill-qwen-14b"),
    "Llama": os.path.join(BASE_DIR, "llama-3.1-8b-instruct"),
    "Qwen2.5": os.path.join(BASE_DIR, "qwen"),
    "Qwen3": os.path.join(BASE_DIR, "qwen3_14b"),
    "Luna": os.path.join(BASE_DIR, "gpt_luna"),
}

# ============================================================
# File map: [model][(format, K)][condition] -> filename (relative to that
# model's directory in MODEL_DIRS). Hardcoded (not templated) because naming
# conventions differ across models/runs. Missing entries stay unset -> reported
# as "file not found" instead of crashing.
# ============================================================
FILES = {
    "DeepSeek-V4": {
        ("Hub", 5):            {"Baseline": "deepseek_v4_scn1_no_perturb.json", "Clinical": "deepseek_v4_scn1_clinical_k5.json", "Random": "deepseek_v4_scn1_random_k5.json"},
        ("Hub", 10):           {"Baseline": "deepseek_v4_scn1_no_perturb.json", "Clinical": "deepseek_v4_scn1_clinical_k10.json", "Random": "deepseek_v4_scn1_random_k10.json"},
        ("hub+Prompt", 5):     {"Baseline": "deepseek_v4_no_pertub_prompt_scn2.json", "Clinical": "deepseek_v4_clinical_prompt_k5.json", "Random": "deepseek_v4_random_prompt_k5.json"},
        ("hub+Prompt", 10):    {"Baseline": "deepseek_v4_no_pertub_prompt_scn2.json", "Clinical": "deepseek_v4_clinical_prompt_k10.json", "Random": "deepseek_v4_random_prompt_k10.json"},
        ("hub+Prompt+Net", 5): {"Baseline": "deepseek_v4_no_perturb_prompt+net_scn3.json", "Clinical": "deepseek_v4_clinical_prompt+net_k5.json", "Random": "deepseek_v4_random_prompt+net_k5.json"},
        ("hub+Prompt+Net", 10):{"Baseline": "deepseek_v4_no_perturb_prompt+net_scn3.json", "Clinical": "deepseek_v4_clinical_prompt+net_k10.json", "Random": "deepseek_v4_random_prompt+net_k10.json"},
    },
    "R1-Distill": {
        ("Hub", 5):            {"Baseline": "deepseek_scn1_no_perturb.json", "Clinical": "deepseek_scn1_clinical_k5.json", "Random": "deepseek_scn1_random_k5.json"},
        ("Hub", 10):           {"Baseline": "deepseek_scn1_no_perturb.json", "Clinical": "deepseek_scn1_clinical_k10.json", "Random": "deepseek_scn1_random_k10.json"},
        ("hub+Prompt", 5):     {"Baseline": "deepseek_no_pertub_prompt.json", "Clinical": "deepseek_clinical_prompt_k5.json", "Random": "deepseek_random_prompt_k5.json"},
        ("hub+Prompt", 10):    {"Baseline": "deepseek_no_pertub_prompt.json", "Clinical": "deepseek_clinical_prompt_k10.json", "Random": "deepseek_random_prompt_k10.json"},
        ("hub+Prompt+Net", 5): {"Baseline": "deepseek_no_perturb_prompt+net.json", "Clinical": "deepseek_clinical_prompt+net_k5.json", "Random": "deepseek_random_prompt+net_k5.json"},
        ("hub+Prompt+Net", 10):{"Baseline": "deepseek_no_perturb_prompt+net.json", "Clinical": "deepseek_clinical_prompt+net_k10.json", "Random": "deepseek_random_prompt+net_k10.json"},
    },
    "Llama": {
        ("Hub", 5):            {"Baseline": "llama_scn1_no_perturb.json", "Clinical": "llama_scn1_clinical_k5.json", "Random": "llama_scn1_random_k5.json"},
        ("Hub", 10):           {"Baseline": "llama_scn1_no_perturb.json", "Clinical": "llama_scn1_clinical_k10.json", "Random": "llama_scn1_random_k10.json"},
        ("hub+Prompt", 5):     {"Baseline": "llama_no_perturb_prompt.json", "Clinical": "llama_clinical_prompt_k5.json", "Random": "llama_random_prompt_k5.json"},
        ("hub+Prompt", 10):    {"Baseline": "llama_no_perturb_prompt.json", "Clinical": "llama_clinical_prompt_k10.json", "Random": "llama_random_prompt_k10.json"},
        ("hub+Prompt+Net", 5): {"Baseline": "llama_no_perturb_prompt+net.json", "Clinical": "llama_clinical_prompt+net_k5.json", "Random": "llama_random_prompt+net_k5.json"},
        ("hub+Prompt+Net", 10):{"Baseline": "llama_no_perturb_prompt+net.json", "Clinical": "llama_clinical_prompt+net_k10.json", "Random": "llama_random_prompt+net_k10.json"},
    },
    "Qwen2.5": {
        ("Hub", 5):            {"Baseline": "qwen_no_perturb_scn1.json", "Clinical": "qwen_clinical_hub_scn1_k5.json", "Random": "qwen_random_hub_scn1_k5.json"},
        ("Hub", 10):           {"Baseline": "qwen_no_perturb_scn1.json", "Clinical": "qwen_clinical_hub_scn1_k10.json", "Random": "qwen_random_hub_scn1_k10.json"},
        ("hub+Prompt", 5):     {"Baseline": "qwen_no_perturb_prompt.json", "Clinical": "qwen_clinical_prompt_k5.json", "Random": "qwen_random_prompt_k5.json"},
        ("hub+Prompt", 10):    {"Baseline": "qwen_no_perturb_prompt.json", "Clinical": "qwen_clinical_prompt_k10.json", "Random": "qwen_random_prompt_k10.json"},
        ("hub+Prompt+Net", 5): {"Baseline": "qwen_no_perturb_prompt+net.json", "Clinical": "qwen_clinical_prompt+net_k5.json", "Random": "qwen_random_prompt+net_k5.json"},
        ("hub+Prompt+Net", 10):{"Baseline": "qwen_no_perturb_prompt+net.json", "Clinical": "qwen_clinical_prompt+net_k10.json", "Random": "qwen_random_prompt+net_k10.json"},
    },
    "Qwen3": {
        ("Hub", 5):            {"Baseline": "qwen3_14b_scn1_no_perturb.json", "Clinical": "qwen3_14b_clinical_k5.json", "Random": "qwen3_14b_scn1_random_k5.json"},
        ("Hub", 10):           {"Baseline": "qwen3_14b_scn1_no_perturb.json", "Clinical": "qwen3_14b_clinical_k10.json", "Random": "qwen3_14b_scn1_random_k10.json"},
        ("hub+Prompt", 5):     {"Baseline": "qwen3_14b_no_pertub_prompt.json", "Clinical": "qwen3_14b_clinical_prompt_k5.json", "Random": "qwen3_14b_random_prompt_k5.json"},
        ("hub+Prompt", 10):    {"Baseline": "qwen3_14b_no_pertub_prompt.json", "Clinical": "qwen3_14b_clinical_prompt_k10.json", "Random": "qwen3_14b_random_prompt_k10.json"},
        ("hub+Prompt+Net", 5): {"Baseline": "qwen3_14b_no_perturb_prompt+net.json", "Clinical": "qwen3_14b_clinical_prompt+net_k5.json", "Random": "qwen3_14b_random_prompt+net_k5.json"},
        ("hub+Prompt+Net", 10):{"Baseline": "qwen3_14b_no_perturb_prompt+net.json", "Clinical": "qwen3_14b_clinical_prompt+net_k10.json", "Random": "qwen3_14b_random_prompt+net_k10.json"},
    },
    "Luna": {
        ("Hub", 5):            {"Baseline": "gpt_scn1_no_perturb.json", "Clinical": "gpt_scn1_clinical_k5.json", "Random": "gpt_scn1_random_k5.json"},
        ("Hub", 10):           {"Baseline": "gpt_scn1_no_perturb.json", "Clinical": "gpt_scn1_clinical_k10.json", "Random": "gpt_scn1_random_k10.json"},
        ("hub+Prompt", 5):     {"Baseline": "gpt_no_pertub_prompt_scn2.json", "Clinical": "gpt_clinical_prompt_k5.json", "Random": "gpt_random_prompt_k5.json"},
        ("hub+Prompt", 10):    {"Baseline": "gpt_no_pertub_prompt_scn2.json", "Clinical": "gpt_clinical_prompt_k10.json", "Random": "gpt_random_prompt_k10.json"},
        ("hub+Prompt+Net", 5): {"Baseline": "gpt_no_perturb_prompt+net_scn3.json", "Clinical": "gpt_clinical_prompt+net_k5.json", "Random": "gpt_random_prompt+net_k5.json"},
        ("hub+Prompt+Net", 10):{"Baseline": "gpt_no_perturb_prompt+net_scn3.json", "Clinical": "gpt_clinical_prompt+net_k10.json", "Random": "gpt_random_prompt+net_k10.json"},
    },
}

COMPARISONS = [("Baseline", "Clinical"), ("Random", "Clinical"), ("Baseline", "Random")]

# ============================================================
# Flip rate + McNemar
# ============================================================
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def flip_and_mcnemar(file_a, file_b, gt):
    a, b = load_json(file_a), load_json(file_b)
    common = set(a) & set(b) & set(gt)
    flips = b_count = c_count = skipped = 0
    for sid in common:
        pred_a, pred_b = a[sid].get("prediction"), b[sid].get("prediction")
        if pred_a not in VALID_LABELS or pred_b not in VALID_LABELS:
            skipped += 1
            continue
        if pred_a != pred_b:
            flips += 1
        correct_a, correct_b = (pred_a == gt[sid]), (pred_b == gt[sid])
        if correct_a and not correct_b:
            b_count += 1
        elif not correct_a and correct_b:
            c_count += 1

    scorable = len(common) - skipped
    flip_rate = flips / scorable if scorable > 0 else float("nan")
    p = mcnemar([[0, b_count], [c_count, 0]], exact=True).pvalue
    return flip_rate, b_count, c_count, p, scorable, skipped

# ============================================================
# Run everything, build one combined lookup table
# ============================================================
rows = []
for model, model_dir in MODEL_DIRS.items():
    for (fmt, k), files in FILES[model].items():
        for cond1, cond2 in COMPARISONS:
            if cond1 not in files or cond2 not in files:
                continue  # e.g. a model with no Clinical files yet -- skip, don't fabricate

            path1 = os.path.join(model_dir, files[cond1])
            path2 = os.path.join(model_dir, files[cond2])

            if not os.path.exists(path1) or not os.path.exists(path2):
                rows.append({
                    "Model": model, "Format": fmt, "K": k,
                    "Comparison": f"{cond1} vs {cond2}",
                    "Flip Rate": None, "b": None, "c": None, "p": None,
                    "Scorable": 0, "Skipped": 0, "Note": "file not found"
                })
                continue

            fr, b_count, c_count, p, scorable, skipped = flip_and_mcnemar(path1, path2, gt)
            rows.append({
                "Model": model, "Format": fmt, "K": k,
                "Comparison": f"{cond1} vs {cond2}",
                "Flip Rate": round(fr, 3), "b": b_count, "c": c_count,
                "p": round(p, 4), "Scorable": scorable, "Skipped": skipped, "Note": ""
            })

results_df = pd.DataFrame(rows)

# ============================================================
# Save + display -- one file, easy to look anything up
# ============================================================
MCNEMAR_CSV = os.path.join(RQ4_DIR, "RQ4_mcnemar_all_models.csv")
results_df.to_csv(MCNEMAR_CSV, index=False)
print(f"Saved {len(results_df)} rows to {MCNEMAR_CSV}\n")

for model in MODEL_DIRS:
    print("=" * 100)
    print(model)
    print("=" * 100)
    display(results_df[results_df["Model"] == model].drop(columns="Model"))


Counter({'Control': 488, 'ADHD': 280})
Saved 108 rows to /home/siu856622573/Code/ADHD code/RQ4/RQ4_mcnemar_all_models.csv

DeepSeek-V4


,Format,K,Comparison,Flip Rate,b,c,p,Scorable,Skipped,Note
0,Hub,5,Baseline vs Clinical,0.468,187,153,0.0734,726,42,
1,Hub,5,Random vs Clinical,0.458,167,153,0.4675,698,70,
2,Hub,5,Baseline vs Random,0.455,176,155,0.2716,728,40,
3,Hub,10,Baseline vs Clinical,0.477,164,186,0.2616,733,35,
4,Hub,10,Random vs Clinical,0.468,152,178,0.1687,705,63,
5,Hub,10,Baseline vs Random,0.470,178,164,0.4821,727,41,
6,hub+Prompt,5,Baseline vs Clinical,0.306,104,118,0.3830,725,43,
7,hub+Prompt,5,Random vs Clinical,0.256,88,103,0.3111,746,22,
8,hub+Prompt,5,Baseline vs Random,0.315,112,111,1.0000,707,61,
9,hub+Prompt,10,Baseline vs Clinical,0.329,106,132,0.1049,723,45,


R1-Distill


,Format,K,Comparison,Flip Rate,b,c,p,Scorable,Skipped,Note
18,Hub,5,Baseline vs Clinical,0.405,175,123,0.0031,736,32,
19,Hub,5,Random vs Clinical,0.461,147,133,0.4373,608,160,
20,Hub,5,Baseline vs Random,0.493,170,135,0.0514,619,149,
21,Hub,10,Baseline vs Clinical,0.389,181,103,0.0000,731,37,
22,Hub,10,Random vs Clinical,0.210,84,68,0.2236,723,45,
23,Hub,10,Baseline vs Random,0.406,179,120,0.0008,737,31,
24,hub+Prompt,5,Baseline vs Clinical,0.495,139,95,0.0048,473,295,
25,hub+Prompt,5,Random vs Clinical,0.494,111,118,0.6918,464,304,
26,hub+Prompt,5,Baseline vs Random,0.511,160,107,0.0014,523,245,
27,hub+Prompt,10,Baseline vs Clinical,0.604,170,106,0.0001,457,311,


Llama


...

,Format,K,Comparison,Flip Rate,b,c,p,Scorable,Skipped,Note
54,Hub,5,Baseline vs Clinical,0.023,11,7,0.4807,768,0,
55,Hub,5,Random vs Clinical,0.030,13,10,0.6776,768,0,
56,Hub,5,Baseline vs Random,0.025,10,9,1.0000,768,0,
57,Hub,10,Baseline vs Clinical,0.023,11,7,0.4807,768,0,
58,Hub,10,Random vs Clinical,0.016,8,4,0.3877,768,0,
59,Hub,10,Baseline vs Random,0.018,7,7,1.0000,768,0,
60,hub+Prompt,5,Baseline vs Clinical,0.031,15,9,0.3075,768,0,
61,hub+Prompt,5,Random vs Clinical,0.059,23,22,1.0000,768,0,
62,hub+Prompt,5,Baseline vs Random,0.033,15,10,0.4244,768,0,
63,hub+Prompt,10,Baseline vs Clinical,0.026,11,9,0.8238,768,0,


Qwen3


,Format,K,Comparison,Flip Rate,b,c,p,Scorable,Skipped,Note
72,Hub,5,Baseline vs Clinical,0.214,82,82,1.0000,768,0,
73,Hub,5,Random vs Clinical,0.402,192,117,0.0000,768,0,
74,Hub,5,Baseline vs Random,0.384,110,185,0.0000,768,0,
75,Hub,10,Baseline vs Clinical,0.185,81,61,0.1105,768,0,
76,Hub,10,Random vs Clinical,0.160,61,62,1.0000,768,0,
77,Hub,10,Baseline vs Random,0.204,89,68,0.1102,768,0,
78,hub+Prompt,5,Baseline vs Clinical,0.585,268,181,0.0000,768,0,
79,hub+Prompt,5,Random vs Clinical,0.388,150,148,0.9538,768,0,
80,hub+Prompt,5,Baseline vs Random,0.548,253,168,0.0000,768,0,
81,hub+Prompt,10,Baseline vs Clinical,0.725,337,220,0.0000,768,0,


Luna


,Format,K,Comparison,Flip Rate,b,c,p,Scorable,Skipped,Note
90,Hub,5,Baseline vs Clinical,0.202,78,77,1.0000,767,1,
91,Hub,5,Random vs Clinical,0.239,97,85,0.4149,762,6,
92,Hub,5,Baseline vs Random,0.186,65,77,0.3560,763,5,
93,Hub,10,Baseline vs Clinical,0.226,88,85,0.8792,766,2,
94,Hub,10,Random vs Clinical,0.171,53,78,0.0356,765,3,
95,Hub,10,Baseline vs Random,0.130,64,36,0.0066,767,1,
96,hub+Prompt,5,Baseline vs Clinical,0.367,179,102,0.0000,766,2,
97,hub+Prompt,5,Random vs Clinical,0.284,123,94,0.0571,765,3,
98,hub+Prompt,5,Baseline vs Random,0.261,124,76,0.0008,767,1,
99,hub+Prompt,10,Baseline vs Clinical,0.436,201,134,0.0003,768,0,


## McNemar summary -- which direction did errors move?

In [29]:
results_df["b"] = pd.to_numeric(results_df["b"], errors="coerce")
results_df["c"] = pd.to_numeric(results_df["c"], errors="coerce")

# b = correct-under-baseline-only (perturbation broke it), c = correct-under-perturbation-only (perturbation fixed it)
b_greater_c = results_df[results_df["b"] > results_df["c"]].copy()

print("Total number of scenarios:", len(results_df))
print("Number of scenarios where b > c (perturbation hurt more than it helped):", len(b_greater_c))
print("Number of scenarios where b <= c:", (results_df["b"] <= results_df["c"]).sum())

print("\nScenarios where b > c:")
print(b_greater_c[["Model", "Format", "K", "Comparison", "b", "c", "p"]].to_string(index=False))

baseline_comparisons = ["Baseline vs Clinical", "Baseline vs Random"]
df_baseline = results_df[results_df["Comparison"].isin(baseline_comparisons)]

summary = (
    df_baseline.groupby("Comparison")
    .apply(lambda x: pd.Series({
        "b > c": (x["b"] > x["c"]).sum(),
        "b < c": (x["b"] < x["c"]).sum(),
        "b = c": (x["b"] == x["c"]).sum(),
        "Total": len(x),
    }), include_groups=False)
)
print("\n", summary)


Total number of scenarios: 108
Number of scenarios where b > c (perturbation hurt more than it helped): 78
Number of scenarios where b <= c: 30

Scenarios where b > c:
      Model         Format  K           Comparison   b   c      p
DeepSeek-V4            Hub  5 Baseline vs Clinical 187 153 0.0734
DeepSeek-V4            Hub  5   Random vs Clinical 167 153 0.4675
DeepSeek-V4            Hub  5   Baseline vs Random 176 155 0.2716
DeepSeek-V4            Hub 10   Baseline vs Random 178 164 0.4821
DeepSeek-V4     hub+Prompt  5   Baseline vs Random 112 111 1.0000
DeepSeek-V4     hub+Prompt 10   Random vs Clinical 108 106 0.9455
DeepSeek-V4 hub+Prompt+Net  5   Baseline vs Random 164 159 0.8239
 R1-Distill            Hub  5 Baseline vs Clinical 175 123 0.0031
 R1-Distill            Hub  5   Random vs Clinical 147 133 0.4373
 R1-Distill            Hub  5   Baseline vs Random 170 135 0.0514
 R1-Distill            Hub 10 Baseline vs Clinical 181 103 0.0000
 R1-Distill            Hub 10   Random v

## 6. Combined heatmap figure -- 3 panels (Baseline vs Clinical, Random vs Clinical, Baseline vs Random)

In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

out_dir = os.path.join(RQ4_DIR, "results")
os.makedirs(out_dir, exist_ok=True)

df = results_df.copy()

# ------------------------------------------------------------
# Font
# ------------------------------------------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["text.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["axes.titlecolor"] = "black"
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"

# ------------------------------------------------------------
# Configuration labels
# ------------------------------------------------------------
abbrev_map = {"Hub": "H", "hub+Prompt": "H+P", "hub+Prompt+Net": "H+P+N"}

df["Model"] = df["Model"].astype(str).str.strip()
df["Config"] = df["Format"].map(abbrev_map) + " K=" + df["K"].astype(str)

col_order = ["H K=5", "H K=10", "H+P K=5", "H+P K=10", "H+P+N K=5", "H+P+N K=10"]
display_labels = ["H\n$K$=5", "H\n$K$=10", "H+P\n$K$=5", "H+P\n$K$=10", "H+P+N\n$K$=5", "H+P+N\n$K$=10"]

model_display = {"R1-Distill": "R1-Distill", "DeepSeek-V4": "DeepSeek-V4", "Luna": "Luna",
                  "Llama": "Llama", "Qwen2.5": "Qwen2.5", "Qwen3": "Qwen3"}
model_order = ["R1-Distill", "DeepSeek-V4", "Luna", "Llama", "Qwen2.5", "Qwen3"]

panel_titles = {
    ("Baseline", "Clinical"): "(a) Baseline vs. Clinical",
    ("Random", "Clinical"):   "(b) Random vs. Clinical",
    ("Baseline", "Random"):   "(c) Baseline vs. Random",
}

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.75), sharey=False,
                          gridspec_kw={"width_ratios": [1, 1, 1], "wspace": 0.08})
cbar_ax = fig.add_axes([0.925, 0.20, 0.014, 0.60])

for i, (cond1, cond2) in enumerate(COMPARISONS):
    comp_label = f"{cond1} vs {cond2}"
    sub = df[df["Comparison"] == comp_label].copy()

    pivot_fr = sub.pivot(index="Model", columns="Config", values="Flip Rate")
    pivot_p = sub.pivot(index="Model", columns="Config", values="p")

    available_models = [m for m in model_order if m in pivot_fr.index]
    pivot_fr = pivot_fr.reindex(available_models)
    pivot_p = pivot_p.reindex(available_models)

    cols = [c for c in col_order if c in pivot_fr.columns]
    pivot_fr = pivot_fr[cols]
    pivot_p = pivot_p[cols]

    annot = pivot_fr.copy().astype(object)
    for r in pivot_fr.index:
        for c in pivot_fr.columns:
            fr = pivot_fr.loc[r, c]
            p = pivot_p.loc[r, c]
            if pd.isna(fr):
                annot.loc[r, c] = ""
            elif p < 0.01:
                annot.loc[r, c] = f"**\n{fr:.2f}"
            elif p < 0.05:
                annot.loc[r, c] = f"*\n{fr:.2f}"
            else:
                annot.loc[r, c] = f"{fr:.2f}"

    sns.heatmap(
        pivot_fr, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1,
        linewidths=0.6, linecolor="white",
        cbar=(i == 2), cbar_ax=cbar_ax if i == 2 else None,
        cbar_kws={"label": "Flip Rate"} if i == 2 else None,
        annot_kws={"fontsize": 8.5, "fontfamily": "Times New Roman", "color": "black",
                   "ha": "center", "va": "center", "linespacing": 0.55},
        ax=axes[i],
    )

    axes[i].set_yticks([j + 0.5 for j in range(len(available_models))])
    if i == 0:
        axes[i].set_yticklabels([model_display.get(m, m) for m in available_models],
                                 rotation=0, fontsize=8, fontfamily="Times New Roman", color="black")
        axes[i].tick_params(axis="y", left=True, labelleft=True, length=0, pad=4, colors="black")
    else:
        axes[i].set_yticklabels([])
        axes[i].tick_params(axis="y", left=False, labelleft=False, colors="black")

    axes[i].set_title(panel_titles[(cond1, cond2)], fontsize=9, fontfamily="Times New Roman",
                       fontweight="normal", color="black", pad=6)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")
    axes[i].set_xticklabels(display_labels, rotation=0, ha="center", fontsize=7.5,
                             fontfamily="Times New Roman", color="black", linespacing=1.0)
    axes[i].tick_params(axis="x", length=0, pad=4, colors="black")

cbar = axes[2].collections[0].colorbar
cbar.ax.tick_params(labelsize=7, length=3, colors="black")
for label in cbar.ax.get_yticklabels():
    label.set_fontfamily("Times New Roman")
    label.set_fontsize(7)
    label.set_color("black")
cbar.set_label("Flip Rate", fontsize=8, fontfamily="Times New Roman", color="black", labelpad=5)

plt.subplots_adjust(left=0.15, right=0.90, top=0.84, bottom=0.27, wspace=0.08)

base = os.path.join(out_dir, "heatmap_combined")
plt.savefig(f"{base}.png", dpi=400, bbox_inches="tight", pad_inches=0.03, facecolor="white")
plt.savefig(f"{base}.pdf", bbox_inches="tight", pad_inches=0.03, facecolor="white")
plt.close()

print(f"Saved:\n{base}.png\n{base}.pdf")
print("\nModels plotted:", available_models)


Saved:
/home/siu856622573/Code/ADHD code/RQ4/results/heatmap_combined.png
/home/siu856622573/Code/ADHD code/RQ4/results/heatmap_combined.pdf

Models plotted: ['R1-Distill', 'DeepSeek-V4', 'Luna', 'Llama', 'Qwen2.5', 'Qwen3']
